# 07 — Text Splitters
### LangChain Retrieval · Part 2 of 4

`06_Document_Loaders/` ended with a number: a 200-page handbook costs ~130,000 tokens *per question*
if you send the whole thing every time. The fix isn't "send less of the document" — it's "send only
the few paragraphs that actually answer this specific question." Before you can retrieve the right
piece, you need pieces. That's what a **text splitter** makes.

> 📌 **What the current research actually says (2026)**
> - A widely-cited 2025 study (Vectara, NAACL) found chunking configuration affects retrieval quality
>   **as much as, or more than, which embedding model you use.** Get the chunk boundaries wrong and a
>   better embedding model won't save you.
> - The practical sweet spot several 2026 benchmarks converge on: **roughly 400–512 tokens per chunk**
>   (~1,600–2,000 characters), with **10–20% overlap**. Too small and a chunk is a fragment with no
>   context; too large and the embedding "blurs" several topics into one vector, so nothing matches it well.
> - **Anthropic's own contextual retrieval research** found that prepending a short piece of
>   context to each chunk *before* embedding it — a sentence explaining what the chunk is about and
>   where it sits in the document — cut retrieval failures by **49%**, and by **67%** combined with a
>   reranker. That's a bigger jump than switching to a more expensive embedding model. The capstone
>   below implements a version of this yourself.
> - `SemanticChunker` lives in `langchain_experimental`, and — same story as `langchain_community` in
>   the last notebook — **that package is also sunset and unmaintained.** It still works and is a
>   useful concept to know (Section 6), but don't build a production pipeline that depends on it.

## Setup

In [ ]:
# %pip install -q langchain langchain-core langchain-text-splitters langchain-groq python-dotenv
# %pip install -q langchain-huggingface sentence-transformers tiktoken
# %pip install -q "langchain-experimental==0.3.4"   # optional — Section 6 only, see the sunset note above


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, Language,
)
print("Environment ready.")


---
## 1. `CharacterTextSplitter` — the naive baseline (and why it's a baseline, not a plan)

Cut every N characters, full stop. It doesn't know what a sentence is, doesn't know what a word is —
it just counts. Seeing it fail is the fastest way to understand what every splitter after this one
is trying to fix.

In [ ]:
SAMPLE_TEXT = (
    "Retrieval-augmented generation combines a language model with an external knowledge source. "
    "Instead of relying only on what the model memorized during training, the system retrieves "
    "relevant passages at query time and includes them in the prompt. This keeps answers grounded "
    "in real, up-to-date information rather than the model's fixed training data."
)

naive_splitter = CharacterTextSplitter(chunk_size=80, chunk_overlap=0, separator="")
naive_chunks = naive_splitter.split_text(SAMPLE_TEXT)

for i, chunk in enumerate(naive_chunks):
    print(f"Chunk {i+1}: {chunk!r}")


Look at chunk boundaries — words get cut in half mid-letter, with no regard for where a sentence
or even a word ends. That's `separator=""`: split anywhere, no exceptions. Passing `separator=" "`
would only cut at spaces, which helps a little (whole words survive) but still cuts sentences and
paragraphs wherever the count happens to land.

---
## 2. `RecursiveCharacterTextSplitter` — the default that actually works

Instead of one separator, this tries a **hierarchy**: paragraph breaks first
(`"\n\n"`), then line breaks (`"\n"`), then sentence-ending punctuation, then spaces, and only as an
absolute last resort, any character. It always prefers the most meaningful break that still keeps
the chunk under your size limit. This is the splitter you should reach for by default.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0)
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1}: {chunk!r}")

print(f"\nNaive splitter:     {len(naive_chunks)} chunks, mid-word cuts")
print(f"Recursive splitter: {len(recursive_chunks)} chunks, breaks at sentence/word boundaries")


### `chunk_overlap` — a deliberately shared strip between neighboring chunks

A single idea often spans a chunk boundary no matter how carefully you split. Overlap repeats the
last N characters of one chunk at the start of the next, so a sentence cut in half still appears in
full in at least one chunk.

In [ ]:
LONG_TEXT = SAMPLE_TEXT + " Retrieval quality often matters as much as which language model you use."

no_overlap = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0).split_text(LONG_TEXT)
with_overlap = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=25).split_text(LONG_TEXT)

print("--- chunk_overlap=0 ---")
for c in no_overlap: print(repr(c))
print("\n--- chunk_overlap=25 ---")
for c in with_overlap: print(repr(c))


Notice the overlapping version repeats the tail end of each chunk at the start of the next — the
`"Retrieval quality often..."` sentence that got split now shows up whole in the chunk after it,
instead of only ever appearing half-cut.

**Rule of thumb from the 2026 benchmarks above:** `chunk_size` around 1,600–2,000 characters
(≈400–512 tokens), `chunk_overlap` at 10–20% of that. Below is what that actually looks like for a
real Document, using `.split_documents()` — the metadata (`source`, `page`, etc.) is copied onto
every resulting chunk automatically.

In [ ]:
policy_doc = Document(
    page_content=SAMPLE_TEXT * 6,   # simulate a longer real document
    metadata={"source": "rag_overview.txt"},
)

production_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = production_splitter.split_documents([policy_doc])

print(f"{len(chunks)} chunks produced")
print(chunks[0].metadata)   # <-- 'source' carried over automatically onto every chunk


---
## 3. Measuring size in **tokens**, not characters

`chunk_size=400` above means 400 *characters* — but the model's context window, and every provider's
pricing, is measured in **tokens**. English averages roughly 4 characters per token, but that ratio
shifts for code, non-English text, and unusual formatting. When precision matters,
`from_tiktoken_encoder()` builds a splitter that counts using an actual tokenizer instead of
guessing from character count.

In [ ]:
try:
    token_aware_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=100,     # this 100 means TOKENS now, not characters
        chunk_overlap=15,
    )
    token_chunks = token_aware_splitter.split_text(SAMPLE_TEXT * 3)
    print(f"{len(token_chunks)} chunks, each capped at 100 TOKENS (not characters)")
    print(f"First chunk length: {len(token_chunks[0])} characters -> ~{len(token_chunks[0])//4} tokens by rough estimate")
except Exception as e:
    # tiktoken downloads its encoding file from openaipublic.blob.core.windows.net on first use —
    # some corporate networks / sandboxes block that URL. If you hit this, either allow that domain
    # once (it caches locally afterward), or just use the ~4-characters-per-token estimate used
    # elsewhere in this notebook — a reasonable approximation for English text.
    print(f"Could not reach tiktoken's encoding download ({type(e).__name__}). "
          f"Falling back to the character-based estimate: ~4 characters per token for English text.")


---
## 4. Structure-aware splitting — when the document already has structure, use it

Recursive splitting guesses at boundaries using generic punctuation. When a document has **known
structure** (Markdown headings, source code), splitting *along that structure* beats guessing.

### Markdown — split by heading, and keep the heading trail as metadata

In [ ]:
markdown_text = """# Support Handbook

## Billing

### Refunds
Refunds are issued within 5-7 business days after we receive the returned item.

### Failed Payments
If a payment fails, the customer has 3 days to update their payment method before the order is cancelled.

## Shipping

### International Orders
International shipping takes 7-14 business days and is available to select countries only.
"""

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(markdown_text)

for chunk in md_chunks:
    print(chunk.metadata, "->", chunk.page_content[:60].replace("\n", " "))


Every chunk now carries `{"h1": "Support Handbook", "h2": "Billing", "h3": "Refunds"}` in its
metadata — so a later answer can cite not just *which file* it came from, but *which section*. This
is often paired with `RecursiveCharacterTextSplitter` as a second pass, for sections still too long.

### Code — split along language-aware boundaries, not generic punctuation

Splitting Python code with a generic splitter can (and regularly does) cut a function definition in
half. `RecursiveCharacterTextSplitter.from_language()` uses separators specific to the language —
splitting before `class`/`def` first, so a function stays whole whenever it fits in one chunk.

**Real-world use:** this is exactly how you'd chunk your own repository's `.py` files to build a
"chat with my codebase" assistant over the projects already in this repo.

In [ ]:
python_code = """
class SupportTicket:
    def __init__(self, subject, body):
        self.subject = subject
        self.body = body

    def summarize(self):
        return f"{self.subject}: {self.body[:50]}"


def classify_urgency(ticket):
    urgent_words = ["refund", "broken", "urgent", "immediately"]
    return "high" if any(w in ticket.body.lower() for w in urgent_words) else "normal"
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=120, chunk_overlap=0,
)
code_chunks = python_splitter.split_text(python_code)

for i, chunk in enumerate(code_chunks):
    print(f"--- chunk {i+1} ---\n{chunk}\n")


Notice the split happened at `def classify_urgency` — a natural function boundary — rather than
mid-way through the `summarize` method, which a generic character splitter would happily do.

---
## 5. Semantic chunking — splitting by *meaning*, not structure

Every splitter so far decides boundaries from **syntax** (character count, punctuation, headings).
`SemanticChunker` decides boundaries from **meaning**: it embeds each sentence, measures how much the
meaning shifts from one sentence to the next, and cuts wherever that shift is largest — so a chunk
stays about one topic even if the document has zero markdown structure to lean on.

The cost: every sentence gets embedded just to *decide where to cut*, before you've even embedded the
final chunks for search. That's why 2026 guidance treats it as a targeted upgrade (heterogeneous,
multi-topic documents), not a default.

> ⚠️ Reference section: `langchain_experimental` is unmaintained (see the note at the top of this
> notebook). The code below runs correctly using our free, local Hugging Face embeddings — but treat
> this as a concept to know, not a production dependency to build on without checking its status first.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

multi_topic_text = """
The Roman Empire expanded across three continents through military conquest and political alliances.
Its road network connected distant provinces and enabled rapid troop movement. Latin became the
administrative language across the territory.

Photosynthesis converts sunlight into chemical energy inside plant chloroplasts. The process
produces oxygen as a byproduct and forms the base of most food chains on Earth.
"""

semantic_splitter = SemanticChunker(
    hf_embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)
semantic_chunks = semantic_splitter.create_documents([multi_topic_text])

print(f"{len(semantic_chunks)} chunks — notice the cut lands between the two topics, not mid-topic:")
for c in semantic_chunks:
    print("---")
    print(c.page_content.strip()[:100], "...")


---
## Real-world capstone: chunking the support knowledge base — with contextual retrieval

Two things, combined: split the knowledge base from `06_Document_Loaders` properly (`chunk_size`
in the recommended range, real overlap), then implement a lightweight version of **Anthropic's
contextual retrieval** — prepend a one-sentence summary of *what this chunk is about* before
embedding it, and measure the difference it makes with the cosine-similarity check you already know
from `01_Models/02_embedding_models.ipynb`.

In [ ]:
support_doc = Document(
    page_content="""Refunds are issued to the original payment method within 5-7 business days
after we receive the returned item. Shipping fees are non-refundable unless the item arrived
damaged. Digital products and gift cards cannot be returned once delivered. Failed payments give
the customer 3 days to update their payment method before the order is automatically cancelled.
International shipping takes 7-14 business days and is available to select countries only.
Standard shipping takes 3-5 business days, and orders over $50 qualify for free standard shipping.""",
    metadata={"source": "policies.txt"},
)

splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
raw_chunks = splitter.split_documents([support_doc])

print(f"{len(raw_chunks)} chunks:")
for i, c in enumerate(raw_chunks):
    print(f"  [{i}] {c.page_content!r}")


In [ ]:
# Contextual retrieval: ask a cheap, fast model for a one-sentence header for EACH chunk,
# using the whole document as context — then prepend it before embedding.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

context_prompt = PromptTemplate(
    template="""Document:
{document}

Chunk:
{chunk}

Write ONE short sentence (under 15 words) situating this chunk within the document
(e.g. what topic/section it's from). Return only that sentence.""",
    input_variables=["document", "chunk"],
)
context_chain = context_prompt | ChatGroq(model="llama-3.1-8b-instant", temperature=0) | StrOutputParser()

contextualized_chunks = []
for chunk in raw_chunks:
    header = context_chain.invoke({"document": support_doc.page_content, "chunk": chunk.page_content})
    contextualized_chunks.append(f"{header.strip()}\n{chunk.page_content}")
    print(f"[{header.strip()}]")

print()


In [ ]:
def cosine_similarity(a, b):
    import numpy as np
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "What happens if my payment doesn't go through?"
query_vector = hf_embeddings.embed_query(query)

raw_vectors = hf_embeddings.embed_documents([c.page_content for c in raw_chunks])
contextual_vectors = hf_embeddings.embed_documents(contextualized_chunks)

print(f"{'chunk':<6}{'raw score':>12}{'contextual score':>20}")
for i in range(len(raw_chunks)):
    raw_score = cosine_similarity(query_vector, raw_vectors[i])
    ctx_score = cosine_similarity(query_vector, contextual_vectors[i])
    print(f"{i:<6}{raw_score:>12.3f}{ctx_score:>20.3f}")


Look at the chunk that actually discusses failed payments: adding a one-sentence header that
mentions "payments" or "billing" explicitly usually widens its similarity margin over neighboring
chunks — because the raw chunk alone may not repeat words that overlap with how the customer phrased
the question, while the generated header can restate the topic in exactly those terms. This is the
same 49–67% failure-rate reduction cited at the top of this notebook, at small scale, with numbers
you produced yourself instead of numbers you just read about.

### Try it yourself
1. Change `chunk_size` on `raw_chunks` to something much smaller (e.g. 60) and much larger (e.g. 800)
   — watch how many chunks come out, and whether ideas start getting split mid-sentence again.
2. Run the contextual retrieval cell with a query that's phrased very differently from the document's
   wording (e.g. "my card got declined") and compare the score gap to the example above.
3. Swap `hf_embeddings` for the OpenAI embeddings from `01_Models` and re-run the comparison — does
   the raw-vs-contextual gap change?
4. Try `MarkdownHeaderTextSplitter` on a real README from this repo and inspect the metadata each
   resulting chunk carries.

---
**Next:** `08_Vector_Store/` — these chunks need somewhere to live that supports fast similarity
search over thousands (or millions) of them, instead of the Python list we used here.
